# Episode Viewer
Interactive visualization of few-shot classification episodes from a retweet graph.

In [41]:
# ── Config ───────────────────────────────────────────────────────────────────
GRAPH_PATH   = "/Users/philipp/Downloads/retweet_graph.pt"
N_HOP        = 2      # k-hop neighborhood around episode nodes
SEED         = 42

In [42]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))  # repo root

import torch
import numpy as np
import plotly.graph_objects as go
from collections import defaultdict
from sklearn.decomposition import PCA
from torch_geometric.data import Data
import random

from experiments.sampler import NeighborSampler

rng = random.Random(SEED)
print('loading', GRAPH_PATH)
raw = torch.load(GRAPH_PATH, map_location='cpu', weights_only=False)
print('loaded')
if isinstance(raw, dict):
    x             = raw['x']
    y             = raw['y']
    edge_index    = raw['edge_index']
    label_names   = list(raw.get('label_names', []))
    feature_names = list(raw.get('feature_names', []))
else:  # torch_geometric Data object
    x             = raw.x
    y             = raw.y
    edge_index    = raw.edge_index
    label_names   = list(getattr(raw, 'label_names', []))
    feature_names = list(getattr(raw, 'feature_names', []))

N     = x.shape[0]
x_np  = x.numpy().astype(np.float32)
y_np  = y.numpy()

print(f"Nodes: {N:,}  |  Features: {x_np.shape[1]}  |  Edges: {edge_index.shape[1]:,}")
print(f"Labels ({len(label_names)}): {label_names[:12]}{'...' if len(label_names) > 12 else ''}")
print(f"Labeled nodes: {(y_np >= 0).sum():,} / {N:,}")

print("Building NeighborSampler...", flush=True)
graph   = Data(edge_index=edge_index, num_nodes=N)
sampler = NeighborSampler(graph, num_hops=N_HOP, limit=15)
print("NeighborSampler ready.")

loading /Users/philipp/Downloads/retweet_graph.pt
loaded
Nodes: 78,672  |  Features: 390  |  Edges: 180,928
Labels (2): ['not_conservative', 'conservative']
Labeled nodes: 78,672 / 78,672
Building NeighborSampler...
NeighborSampler ready.


### old plotting helpers

In [ ]:
# ── Adjacency list (used only for NM episode sampling) ───────────────────────
adj = defaultdict(set)
for s, d in zip(edge_index[0].tolist(), edge_index[1].tolist()):
    adj[s].add(d)
    adj[d].add(s)
print("Adjacency list built.")

import networkx as nx

# ── Layout: PCA init + spring refinement ─────────────────────────────────────
def layout_graph(nodes, edges, x_sub=None, seed=SEED, k=None, iterations=80):
    """
    Graph-aware 2D layout.
    Uses PCA as initialization when features are available, then refines with spring layout.
    """
    G = nx.Graph()
    G.add_nodes_from(range(len(nodes)))
    G.add_edges_from(edges)

    init = None
    if x_sub is not None and len(nodes) >= 2:
        try:
            pca_pos = layout_pca(x_sub)
            init = {i: pca_pos[i] for i in range(len(nodes))}
        except Exception:
            init = None

    pos_dict = nx.spring_layout(
        G,
        pos=init,
        seed=seed,
        k=k,
        iterations=iterations,
        weight=None,
    )

    return np.array([pos_dict[i] for i in range(len(nodes))])


# ── Classification episode sampler ───────────────────────────────────────────
def sample_episode(n_way, n_shot, n_query):
    by_label = defaultdict(list)
    for i, lbl in enumerate(y_np):
        if lbl >= 0:
            by_label[int(lbl)].append(i)
    eligible = [l for l, ns in by_label.items() if len(ns) >= n_shot + n_query]
    if len(eligible) < n_way:
        raise ValueError(f"Only {len(eligible)} classes have >= {n_shot+n_query} nodes; need {n_way}.")
    chosen = rng.sample(eligible, n_way)
    episode = {}
    for lbl in chosen:
        ns = by_label[lbl].copy()
        rng.shuffle(ns)
        episode[lbl] = {'support': ns[:n_shot], 'query': ns[n_shot:n_shot + n_query], 'center': None}
    return episode


# ── Neighbor-matching episode sampler ─────────────────────────────────────────
def sample_episode_nm(n_way, n_shot, n_query):
    needed  = n_shot + n_query
    centers = []
    pool    = list(range(N))
    rng.shuffle(pool)
    for candidate in pool:
        if len(adj[candidate]) >= needed:
            centers.append(candidate)
        if len(centers) == n_way:
            break
    if len(centers) < n_way:
        raise ValueError(f"Only {len(centers)} nodes have >= {needed} neighbors; need {n_way}.")
    episode = {}
    for center in centers:
        neighbors = list(adj[center])
        rng.shuffle(neighbors)
        episode[center] = {
            'support': neighbors[:n_shot],
            'query':   neighbors[n_shot:n_shot + n_query],
            'center':  center,
        }
    return episode


# ── Subgraph via actual model sampler ─────────────────────────────────────────
def k_hop_subgraph(seed_nodes):
    """
    Call sampler.sample_node per seed node (exactly as the model does),
    then union the resulting subgraphs.
    """
    global_nodes = set()
    global_edges = set()

    for node in seed_nodes:
        node_list, edge_idx, _ = sampler.sample_node(node)
        nl = node_list.tolist()
        global_nodes.update(nl)
        for s, d in zip(edge_idx[0].tolist(), edge_idx[1].tolist()):
            gs, gd = nl[s], nl[d]
            global_edges.add((min(gs, gd), max(gs, gd)))

    nodes = sorted(global_nodes)
    n2l   = {n: i for i, n in enumerate(nodes)}
    edges = [(n2l[a], n2l[b]) for a, b in global_edges]
    return nodes, edges, n2l


# ── 2D layout via PCA ────────────────────────────────────────────────────────
def layout_pca(x_sub):
    emb_cols = [i for i, n in enumerate(feature_names) if n.startswith('emb_')]
    src = x_sub[:, emb_cols] if emb_cols else x_sub
    if src.shape[1] >= 2:
        return PCA(n_components=2, random_state=SEED).fit_transform(src)
    pad = np.zeros((src.shape[0], 2 - src.shape[1]))
    return np.hstack([src, pad])


# ── Hover text ───────────────────────────────────────────────────────────────
def node_hover(global_idx, role, lbl_name, x_row, max_stats=10):
    lines = [f'<b>{role}</b>', f'Node id: {global_idx}']
    if lbl_name:
        lines.append(f'Label: <b>{lbl_name}</b>')
    stats = [(i, n) for i, n in enumerate(feature_names) if not n.startswith('emb_')]
    stats += [(i, n) for i, n in enumerate(feature_names) if n.startswith('emb_')][:3]
    if stats:
        lines.append('─' * 22)
        for i, name in stats[:max_stats]:
            lines.append(f'{name}: {x_row[i]:.4g}')
        if len(stats) > max_stats:
            lines.append(f'… +{len(stats) - max_stats} more features')
    elif len(x_row) > 0:
        lines.append(f'Embedding L2: {float(np.linalg.norm(x_row)):.4f}')
    return '<br>'.join(lines)


print("Helpers ready.")

# ── Visualization ─────────────────────────────────────────────────────────────
COLORS = [
    '#e41a1c', '#377eb8', '#4daf4a', '#984ea3',
    '#ff7f00', '#a65628', '#f781bf', '#17becf',
]

def visualize_episode(episode, ep_idx, task="classification"):
    label_ids   = sorted(episode.keys())
    color_map   = {lbl: COLORS[i % len(COLORS)] for i, lbl in enumerate(label_ids)}
    support_map = {n: lbl for lbl, sets in episode.items() for n in sets['support']}
    query_map   = {n: lbl for lbl, sets in episode.items() for n in sets['query']}
    center_map  = {sets['center']: lbl for lbl, sets in episode.items() if sets['center'] is not None}
    seed_set    = set(support_map) | set(query_map) | set(center_map)

    nodes, edges, n2l = k_hop_subgraph(sorted(seed_set))
    pos = layout_pca(x_np[nodes])

    fig = go.Figure()

    # edges
    ex, ey = [], []
    for a, b in edges:
        ex += [pos[a, 0], pos[b, 0], None]
        ey += [pos[a, 1], pos[b, 1], None]
    fig.add_trace(go.Scatter(
        x=ex, y=ey, mode='lines',
        line=dict(color='#cccccc', width=0.6),
        hoverinfo='none', showlegend=False,
    ))

    # background nodes
    bg = [i for i, n in enumerate(nodes) if n not in seed_set]
    if bg:
        fig.add_trace(go.Scatter(
            x=pos[bg, 0], y=pos[bg, 1], mode='markers',
            marker=dict(size=5, color='#d8d8d8', line=dict(width=0)),
            text=[node_hover(nodes[i], 'Background', '', x_np[nodes[i]]) for i in bg],
            hovertemplate='%{text}<extra></extra>',
            name='Background', legendgroup='bg',
        ))

    # per-class: support, query, and (for NM) center
    for lbl in label_ids:
        color    = color_map[lbl]
        lbl_name = (label_names[lbl] if lbl < len(label_names) else str(lbl)) if task == "classification" else f'Center {lbl}'

        for role, node_map, symbol, size in [
            ('Support', support_map, 'circle',  15),
            ('Query',   query_map,   'diamond', 15),
            ('Center',  center_map,  'star',    20),
        ]:
            idxs = [n2l[n] for n, l in node_map.items() if l == lbl and n in n2l]
            if not idxs:
                continue
            fig.add_trace(go.Scatter(
                x=pos[idxs, 0], y=pos[idxs, 1], mode='markers',
                marker=dict(size=size, color=color, symbol=symbol,
                            line=dict(color='black', width=1.5)),
                text=[node_hover(nodes[i], f'{role} – {lbl_name}', lbl_name, x_np[nodes[i]])
                      for i in idxs],
                hovertemplate='%{text}<extra></extra>',
                name=f'{lbl_name} ({role})', legendgroup=f'cls_{lbl}',
            ))

    if task == "classification":
        ep_labels = ', '.join(label_names[l] if l < len(label_names) else str(l) for l in label_ids)
        title = f'Episode {ep_idx + 1}  —  {ep_labels}'
    else:
        title = f'Episode {ep_idx + 1}  —  NM  (centers: {", ".join(str(l) for l in label_ids)})'

    fig.update_layout(
        title=dict(text=title, font=dict(size=14)),
        xaxis=dict(visible=False), yaxis=dict(visible=False),
        plot_bgcolor='white',
        hovermode='closest',
        legend=dict(itemsizing='constant', font=dict(size=11)),
        width=950, height=640,
        margin=dict(l=20, r=20, t=50, b=20),
    )
    return fig

Adjacency list built.
Helpers ready.


### plotting helpers

In [ ]:
from collections import defaultdict
import numpy as np
import plotly.graph_objects as go
from sklearn.decomposition import PCA

try:
    import networkx as nx
    HAS_NX = True
except ImportError:
    HAS_NX = False


# ── Adjacency list used for NM episode sampling ──────────────────────────────
adj = defaultdict(set)

for s, d in zip(edge_index[0].tolist(), edge_index[1].tolist()):
    adj[s].add(d)
    adj[d].add(s)

print("Adjacency list built.")


# ── Classification episode sampler ───────────────────────────────────────────
def sample_episode(n_way, n_shot, n_query):
    by_label = defaultdict(list)

    for i, lbl in enumerate(y_np):
        if lbl >= 0:
            by_label[int(lbl)].append(i)

    needed = n_shot + n_query
    eligible = [lbl for lbl, nodes in by_label.items() if len(nodes) >= needed]

    if len(eligible) < n_way:
        raise ValueError(
            f"Only {len(eligible)} classes have >= {needed} nodes; need {n_way}."
        )

    chosen = rng.sample(eligible, n_way)
    episode = {}

    for lbl in chosen:
        nodes = by_label[lbl].copy()
        rng.shuffle(nodes)

        episode[lbl] = {
            "support": nodes[:n_shot],
            "query": nodes[n_shot:n_shot + n_query],
            "center": None,
        }

    return episode


# ── Neighbor-matching episode sampler ────────────────────────────────────────
def sample_episode_nm(n_way, n_shot, n_query):
    needed = n_shot + n_query
    centers = []

    pool = list(range(N))
    rng.shuffle(pool)

    for candidate in pool:
        if len(adj[candidate]) >= needed:
            centers.append(candidate)

        if len(centers) == n_way:
            break

    if len(centers) < n_way:
        raise ValueError(
            f"Only {len(centers)} nodes have >= {needed} neighbors; need {n_way}."
        )

    episode = {}

    for center in centers:
        neighbors = list(adj[center])
        rng.shuffle(neighbors)

        episode[center] = {
            "support": neighbors[:n_shot],
            "query": neighbors[n_shot:n_shot + n_query],
            "center": center,
        }

    return episode


# ── Subgraph via actual model sampler ────────────────────────────────────────
def k_hop_subgraph(seed_nodes):
    """
    Calls sampler.sample_node per seed node exactly as the model does,
    unions the resulting node sets, then looks up all edges within those
    nodes from the full adj — so bg-to-bg connections are included.
    """
    global_nodes = set()

    for node in seed_nodes:
        node_list, _, _ = sampler.sample_node(node)
        global_nodes.update(node_list.tolist())

    nodes = sorted(global_nodes)
    n2l = {node: i for i, node in enumerate(nodes)}

    edges = set()
    for n in nodes:
        for nb in adj[n]:
            if nb in n2l:
                edges.add((min(n2l[n], n2l[nb]), max(n2l[n], n2l[nb])))

    return nodes, list(edges), n2l


# ── Layout helpers ───────────────────────────────────────────────────────────
def layout_pca(x_sub):
    """
    Feature-space layout. Good for seeing embedding similarity.
    """
    emb_cols = [
        i for i, name in enumerate(feature_names)
        if name.startswith("emb_")
    ]

    src = x_sub[:, emb_cols] if emb_cols else x_sub

    if src.shape[1] >= 2:
        pos = PCA(n_components=2, random_state=SEED).fit_transform(src)
    else:
        pad = np.zeros((src.shape[0], 2 - src.shape[1]))
        pos = np.hstack([src, pad])

    return normalize_pos(pos)


def layout_graph(nodes, edges, x_sub=None, seed=SEED, k=None, iterations=120):
    """
    Graph-aware layout. Usually better for visualizing sampled neighborhoods.
    Uses PCA as initialization when possible.
    """
    if not HAS_NX or len(nodes) == 0:
        return layout_pca(x_sub) if x_sub is not None else np.zeros((len(nodes), 2))

    G = nx.Graph()
    G.add_nodes_from(range(len(nodes)))
    G.add_edges_from(edges)

    if x_sub is not None and len(nodes) >= 2:
        init = layout_pca(x_sub)
        init_dict = {i: init[i] for i in range(len(nodes))}
    else:
        init_dict = None

    if k is None:
        k = 1.8 / np.sqrt(max(len(nodes), 1))

    pos_dict = nx.spring_layout(
        G,
        pos=init_dict,
        seed=seed,
        k=k,
        iterations=iterations,
        weight=None,
    )

    pos = np.array([pos_dict[i] for i in range(len(nodes))])
    return normalize_pos(pos)


def normalize_pos(pos):
    """
    Centers and rescales a 2D layout for consistent rendering.
    """
    pos = np.asarray(pos, dtype=float)

    if len(pos) == 0:
        return pos.reshape(0, 2)

    pos = pos - pos.mean(axis=0, keepdims=True)

    scale = np.abs(pos).max()
    if scale > 0:
        pos = pos / scale

    return pos


# ── Hover text ───────────────────────────────────────────────────────────────
def node_hover(global_idx, role, lbl_name, x_row, max_stats=10):
    lines = [
        f"<b>{role}</b>",
        f"Node id: {global_idx}",
    ]

    if lbl_name:
        lines.append(f"Label: <b>{lbl_name}</b>")

    stats = [
        (i, name)
        for i, name in enumerate(feature_names)
        if not name.startswith("emb_")
    ]

    stats += [
        (i, name)
        for i, name in enumerate(feature_names)
        if name.startswith("emb_")
    ][:3]

    if stats:
        lines.append("─" * 22)

        for i, name in stats[:max_stats]:
            lines.append(f"{name}: {x_row[i]:.4g}")

        if len(stats) > max_stats:
            lines.append(f"… +{len(stats) - max_stats} more features")

    elif len(x_row) > 0:
        lines.append(f"Embedding L2: {float(np.linalg.norm(x_row)):.4f}")

    return "<br>".join(lines)


print("Helpers ready.")


# ── Visualization ────────────────────────────────────────────────────────────
COLORS = [
    "#e41a1c", "#377eb8", "#4daf4a", "#984ea3",
    "#ff7f00", "#a65628", "#f781bf", "#17becf",
]


def visualize_episode(
    episode,
    ep_idx,
    task="classification",
    layout="graph",
    show_background=True,
    width=1050,
    height=720,
    k=None
):
    """
    layout:
        "graph" -> graph-aware spring layout, best for topology
        "pca"   -> feature-space PCA layout, best for embedding similarity
    """
    label_ids = sorted(episode.keys())
    color_map = {
        lbl: COLORS[i % len(COLORS)]
        for i, lbl in enumerate(label_ids)
    }

    support_map = {
        node: lbl
        for lbl, sets in episode.items()
        for node in sets["support"]
    }

    query_map = {
        node: lbl
        for lbl, sets in episode.items()
        for node in sets["query"]
    }

    center_map = {
        sets["center"]: lbl
        for lbl, sets in episode.items()
        if sets["center"] is not None
    }

    seed_set = set(support_map) | set(query_map) | set(center_map)

    nodes, edges, n2l = k_hop_subgraph(sorted(seed_set))
    x_sub = x_np[nodes]

    if layout == "pca":
        pos = layout_pca(x_sub)
        layout_name = "PCA feature layout"
    elif layout == "graph":
        pos = layout_graph(nodes, edges, x_sub=x_sub)
        layout_name = "graph layout"
    else:
        raise ValueError("layout must be either 'graph' or 'pca'.")

    fig = go.Figure()

    # Edges
    if edges:
        ex, ey = [], []

        for a, b in edges:
            ex.extend([pos[a, 0], pos[b, 0], None])
            ey.extend([pos[a, 1], pos[b, 1], None])

        fig.add_trace(
            go.Scatter(
                x=ex,
                y=ey,
                mode="lines",
                line=dict(color="rgba(160,160,160,0.35)", width=0.8),
                hoverinfo="none",
                showlegend=False,
            )
        )

    # Background nodes
    bg = [
        i for i, node in enumerate(nodes)
        if node not in seed_set
    ]

    if show_background and bg:
        fig.add_trace(
            go.Scatter(
                x=pos[bg, 0],
                y=pos[bg, 1],
                mode="markers",
                marker=dict(
                    size=5,
                    color="rgba(180,180,180,0.45)",
                    line=dict(width=0),
                ),
                text=[
                    node_hover(nodes[i], "Background", "", x_np[nodes[i]])
                    for i in bg
                ],
                hovertemplate="%{text}<extra></extra>",
                name="Background",
                legendgroup="background",
                opacity=0.8,
            )
        )

    # Episode nodes
    role_specs = [
        ("Support", support_map, "circle", 16),
        ("Query", query_map, "diamond", 16),
        ("Center", center_map, "star", 24),
    ]

    for lbl in label_ids:
        color = color_map[lbl]

        if task == "classification":
            lbl_name = label_names[lbl] if lbl < len(label_names) else str(lbl)
        else:
            lbl_name = f"Center {lbl}"

        for role, node_map, symbol, size in role_specs:
            idxs = [
                n2l[node]
                for node, node_lbl in node_map.items()
                if node_lbl == lbl and node in n2l
            ]

            if not idxs:
                continue

            fig.add_trace(
                go.Scatter(
                    x=pos[idxs, 0],
                    y=pos[idxs, 1],
                    mode="markers",
                    marker=dict(
                        size=size,
                        color=color,
                        symbol=symbol,
                        line=dict(color="black", width=1.4),
                    ),
                    text=[
                        node_hover(
                            nodes[i],
                            f"{role} – {lbl_name}",
                            lbl_name,
                            x_np[nodes[i]],
                        )
                        for i in idxs
                    ],
                    hovertemplate="%{text}<extra></extra>",
                    name=f"{lbl_name} · {role}",
                    legendgroup=f"class_{lbl}",
                )
            )

    # Optional center-to-neighbor emphasis for NM
    if task != "classification":
        for lbl, sets in episode.items():
            center = sets["center"]
            if center not in n2l:
                continue

            cx, cy = pos[n2l[center]]

            for node in sets["support"] + sets["query"]:
                if node not in n2l:
                    continue

                nx_, ny_ = pos[n2l[node]]

                fig.add_trace(
                    go.Scatter(
                        x=[cx, nx_],
                        y=[cy, ny_],
                        mode="lines",
                        line=dict(
                            color=color_map[lbl],
                            width=1.2,
                            dash="dot",
                        ),
                        hoverinfo="none",
                        showlegend=False,
                        opacity=0.45,
                    )
                )

    if task == "classification":
        ep_labels = ", ".join(
            label_names[lbl] if lbl < len(label_names) else str(lbl)
            for lbl in label_ids
        )
        title = f"Episode {ep_idx + 1} — {ep_labels}"
    else:
        centers = ", ".join(str(lbl) for lbl in label_ids)
        title = f"Episode {ep_idx + 1} — Neighbor matching — centers: {centers}"

    fig.update_layout(
        title=dict(
            text=f"{title}<br><sup>{layout_name}; {len(nodes)} nodes, {len(edges)} edges</sup>",
            font=dict(size=15),
        ),
        plot_bgcolor="white",
        paper_bgcolor="white",
        hovermode="closest",
        width=width,
        height=height,
        margin=dict(l=20, r=220, t=70, b=20),
        legend=dict(
            x=1.02,
            y=1.0,
            xanchor="left",
            yanchor="top",
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="rgba(0,0,0,0.12)",
            borderwidth=1,
            font=dict(size=11),
            itemsizing="constant",
        ),
    )

    fig.update_xaxes(
        visible=False,
        scaleanchor="y",
        scaleratio=1,
        zeroline=False,
    )

    fig.update_yaxes(
        visible=False,
        zeroline=False,
    )

    return fig

In [49]:
TASK         = "classification"   # "classification"  or  "neighbor_matching"
N_EPISODES   = 3      # number of episodes to visualize
N_WAY        = 2      # classes per episode
N_SHOT       = 1      # support nodes per class
N_QUERY      = 1      # query nodes per class


# ── Run ──────────────────────────────────────────────────────────────────────
for ep_idx in range(N_EPISODES):
    if TASK == "neighbor_matching":
        episode = sample_episode_nm(N_WAY, N_SHOT, N_QUERY)
    else:
        episode = sample_episode(N_WAY, N_SHOT, N_QUERY)
    fig = visualize_episode(episode, ep_idx, task=TASK)
    fig.show()